# Data Acquisition
**COSC 301 Project — Malaysia State-Level Socioeconomic & Health Outcomes**

This notebook downloads raw data from three sources:
1. **OpenDOSM** : household income, poverty, and population by state
2. **MoH Malaysia GitHub** : public health facilities and hospital beds
3. **World Bank API** : national-level health indicators (backup / cross-validation)

All files are saved to `data/raw/`.

In [1]:
import requests
import pandas as pd
import os
import time

## Setup

In [2]:
RAW_DIR = "data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

## Helper Functions

In [3]:
def fetch_json(url, params=None, retries=3, delay=2):
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=15)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(delay)
            else:
                raise


def fetch_csv(url, save_path, retries=3, delay=2):
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            with open(save_path, "wb") as f:
                f.write(r.content)
            return
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(delay)
            else:
                raise

## Source 1: OpenDOSM (`api.data.gov.my`)

**License:** Creative Commons Attribution (CC-BY)  
**Docs:** https://open.dosm.gov.my/api-docs

| Dataset | Description |
|---|---|
| `hh_income_parlimen` | Household income by parliament constituency |
| `hh_poverty_parlimen` | Household poverty by parliament constituency |
| `population_state` | State-level population totals (overall age/sex/ethnicity only) for per-capita normalisation |

In [4]:
BASE_DOSM = "https://api.data.gov.my/data-catalogue"

DOSM_DATASETS = [
    {"id": "hh_income_parlimen",  "save_as": "hh_income_parlimen.csv",  "limit": 2000},
    {"id": "hh_poverty_parlimen", "save_as": "hh_poverty_parlimen.csv", "limit": 2000},
    {"id": "population_state",    "save_as": "population_state.csv",    "limit": 10000,
     "filter": "age@overall_age,sex@overall_sex,ethnicity@overall_ethnicity"},
]

for ds in DOSM_DATASETS:
    params = {"id": ds["id"], "limit": ds["limit"]}
    if "filter" in ds:
        params["filter"] = ds["filter"]
    df = pd.DataFrame(fetch_json(BASE_DOSM, params=params))
    df.to_csv(os.path.join(RAW_DIR, ds["save_as"]), index=False)
    print(f"{ds['id']}: {len(df)} rows saved")

hh_income_parlimen: 666 rows saved
hh_poverty_parlimen: 666 rows saved


HTTPError: 400 Client Error: Bad Request for url: https://api.data.gov.my/data-catalogue/?id=population_state&limit=10000&filter=age%40overall_age%2Csex%40overall_sex%2Cethnicity%40overall_ethnicity

## Source 2: MoH Malaysia GitHub

**License:** Malaysian Open Data License (free for research)  
**Docs:** https://github.com/MoH-Malaysia

MoH publishes data directly on GitHub as CSVs. Pulling from the official repo is more reliable than scraping the portal.

| File | Description |
|---|---|
| `moh_facilities.csv` | Public hospitals and health clinics by state |
| `moh_beds.csv` | Hospital bed counts and utilisation by facility |

In [ ]:
MOH_BASE = "https://raw.githubusercontent.com/MoH-Malaysia/data-resources-public/main"

MOH_DATASETS = [
    {"path": "facilities_master.csv", "save_as": "moh_facilities.csv"},
    {"path": "bedutil_facility.csv",  "save_as": "moh_beds.csv"},
]

for ds in MOH_DATASETS:
    save_path = os.path.join(RAW_DIR, ds["save_as"])
    fetch_csv(f"{MOH_BASE}/{ds['path']}", save_path)
    df = pd.read_csv(save_path)
    print(f"{ds['save_as']}: {len(df)} rows saved")

## Source 3: World Bank API (National-Level Backup)

**License:** CC-BY 4.0  
**Coverage:** Malaysia (`MYS`), most recent 20 years

This is national-level 

| Indicator | Column | Description |
|---|---|---|
| `SP.DYN.LE00.IN` | `life_expectancy` | Life expectancy at birth |
| `SP.DYN.IMRT.IN` | `infant_mortality` | Infant mortality (per 1,000 live births) |
| `SH.XPD.CHEX.GD.ZS` | `health_exp_gdp` | Health expenditure (% of GDP) |
| `SH.MED.BEDS.ZS` | `hospital_beds` | Hospital beds (per 1,000 people) |

In [ ]:
WB_BASE = "https://api.worldbank.org/v2/country/MYS/indicator"

WB_INDICATORS = [
    ("SP.DYN.LE00.IN",     "life_expectancy"),
    ("SP.DYN.IMRT.IN",     "infant_mortality"),
    ("SH.XPD.CHEX.GD.ZS", "health_exp_gdp"),
    ("SH.MED.BEDS.ZS",     "hospital_beds"),
]

wb_frames = []
for code, col in WB_INDICATORS:
    data = fetch_json(f"{WB_BASE}/{code}", params={"format": "json", "per_page": 100, "mrv": 20})
    records = data[1] if isinstance(data, list) and len(data) > 1 else []
    rows = [{"year": int(r["date"]), col: r["value"]} for r in records if r["value"] is not None]
    if rows:
        wb_frames.append(pd.DataFrame(rows).set_index("year"))

wb_df = pd.concat(wb_frames, axis=1).reset_index().sort_values("year")
wb_df.to_csv(os.path.join(RAW_DIR, "worldbank_malaysia.csv"), index=False)
print(f"World Bank: {len(wb_df)} rows saved")

## Validation: State Name Consistency

Checks that state names across all sources match the 16 official DOSM state/territory names.  
Mismatches will break joins in `clean_data` — any `WARN` here should be added to `STATE_NAME_MAP` in that script.

In [5]:
OFFICIAL_STATES = [
    "Johor", "Kedah", "Kelantan", "Melaka", "Negeri Sembilan",
    "Pahang", "Perak", "Perlis", "Pulau Pinang", "Sabah",
    "Sarawak", "Selangor", "Terengganu",
    "W.P. Kuala Lumpur", "W.P. Labuan", "W.P. Putrajaya",
]

for name, fname in [
    ("hh_income_parlimen",  "hh_income_parlimen.csv"),
    ("hh_poverty_parlimen", "hh_poverty_parlimen.csv"),
    ("moh_facilities",      "moh_facilities.csv"),
    ("moh_beds",            "moh_beds.csv"),
]:
    path = os.path.join(RAW_DIR, fname)
    if not os.path.exists(path):
        print(f"SKIP  {name}: file not found")
        continue
    df = pd.read_csv(path)
    if "state" not in df.columns:
        continue
    found = set(df["state"].unique())
    mismatches = found - set(OFFICIAL_STATES)
    missing = set(OFFICIAL_STATES) - found
    if not mismatches and not missing:
        print(f"OK    {name}: all 16 states match")
    else:
        if mismatches:
            print(f"WARN  {name}: unexpected states: {mismatches}")
        if missing:
            print(f"WARN  {name}: missing states: {missing}")

OK    hh_income_parlimen: all 16 states match
OK    hh_poverty_parlimen: all 16 states match
OK    moh_facilities: all 16 states match
OK    moh_beds: all 16 states match
